# ISIC 2018 CAM Analysis Example

This notebook demonstrates how to use the research repository for CAM generation and evaluation.

In [ ]:
import sys
import os

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## Load Configuration

In [ ]:
from src.utils.config import load_all_configs

configs = load_all_configs('../configs')
paths_config = configs['paths']
model_config = configs['model']
eval_config = configs['eval']

print("ISIC 2018 Classes:")
for i, cls in enumerate(paths_config['classes']):
    print(f"  {i}: {cls}")

## Load Model and Data

In [ ]:
from src.models.siim_inference import SIIMModelWrapper
from src.data.isic_dataset import create_dataloader

# Load model
model_path = paths_config['external']['siim']['model_path']
model_wrapper = SIIMModelWrapper(
    model_path=model_path,
    device=model_config['inference']['device']
)

# Load data
dataloader = create_dataloader(
    csv_file=paths_config['data']['isic2018']['test_split'],
    img_dir=paths_config['data']['isic2018']['images'],
    batch_size=4,
    shuffle=False,
    num_workers=2,
    input_size=model_config['siim_model']['input_size'],
    augment=False,
    class_names=paths_config['classes']
)

print(f"Model loaded on: {model_wrapper.device}")
print(f"Dataset size: {len(dataloader.dataset)}")

## Run Inference on Sample Batch

In [ ]:
# Get a batch
batch = next(iter(dataloader))
images = batch['image']
image_names = batch['image_name']

# Run inference
preds, probs = model_wrapper.predict(images)

# Display results
print("\nPredictions:")
for i in range(len(images)):
    pred_class = preds[i].item()
    pred_name = paths_config['classes'][pred_class]
    confidence = probs[i, pred_class].item()
    print(f"{image_names[i]}: {pred_name} (confidence: {confidence:.3f})")

## Generate CAM Visualization

In [ ]:
from src.utils.visualization import create_overlay, tensor_to_numpy_image, normalize_cam

# Get target layer
target_layer = model_wrapper.get_target_layer()
model_wrapper.register_hooks(target_layer)

# Generate CAM for first image
image = images[0:1].to(model_wrapper.device)
target_class = preds[0].item()

# Forward pass
image.requires_grad = True
logits = model_wrapper.model(image)

# Backward pass
model_wrapper.model.zero_grad()
logits[0, target_class].backward()

# Compute CAM
gradients = model_wrapper.gradients[0].cpu().numpy()
features = model_wrapper.features[0].cpu().numpy()
weights = np.mean(gradients, axis=(1, 2))
cam = np.sum(weights[:, None, None] * features, axis=0)
cam = np.maximum(cam, 0)

# Visualize
img_np = tensor_to_numpy_image(images[0])
overlay = create_overlay(img_np, cam, alpha=0.5, colormap='jet')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_np)
axes[0].set_title(f'Original Image')
axes[0].axis('off')

axes[1].imshow(normalize_cam(cam), cmap='jet')
axes[1].set_title(f'CAM Heatmap')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title(f'Overlay (Class: {paths_config["classes"][target_class]})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Evaluate CAM with Confidence Drop

In [ ]:
from src.eval.cam_metrics import ConfidenceDropMetric
import torch.nn.functional as F

# Initialize metric
metric = ConfidenceDropMetric(
    model_wrapper.model,
    perturbation_steps=[0.1, 0.3, 0.5, 0.7, 0.9]
)

# Resize CAM to match image size
cam_tensor = torch.tensor(cam).unsqueeze(0).unsqueeze(1).float()
cam_resized = F.interpolate(
    cam_tensor,
    size=images.shape[-2:],
    mode='bilinear',
    align_corners=False
).squeeze(1)

# Compute metric
result = metric.compute(
    images[0:1].to(model_wrapper.device),
    cam_resized.to(model_wrapper.device),
    preds[0:1].to(model_wrapper.device)
)

# Plot results
ratios = [d['ratio'] for d in result['drops']]
confidences = [d['confidence'] for d in result['drops']]

plt.figure(figsize=(10, 6))
plt.plot(ratios, confidences, marker='o', linewidth=2, markersize=8)
plt.axhline(y=result['original_confidence'], color='r', linestyle='--', label='Original')
plt.xlabel('Perturbation Ratio', fontsize=12)
plt.ylabel('Confidence', fontsize=12)
plt.title('Confidence Drop Analysis', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nAverage Confidence Drop: {result['average_drop']:.4f}")
print(f"Relative Drop: {result['average_relative_drop']:.4f}")

## Generate Differential CAM

In [ ]:
# Get top-2 predictions
top2_probs, top2_classes = torch.topk(probs[0], 2)
class1, class2 = top2_classes[0].item(), top2_classes[1].item()

print(f"Top-2 classes:")
print(f"  1. {paths_config['classes'][class1]} ({top2_probs[0].item():.3f})")
print(f"  2. {paths_config['classes'][class2]} ({top2_probs[1].item():.3f})")

# Generate CAMs for both classes
def generate_cam_for_class(image, target_class):
    image.requires_grad = True
    logits = model_wrapper.model(image)
    model_wrapper.model.zero_grad()
    logits[0, target_class].backward(retain_graph=True)
    
    gradients = model_wrapper.gradients[0].cpu().numpy()
    features = model_wrapper.features[0].cpu().numpy()
    weights = np.mean(gradients, axis=(1, 2))
    cam = np.sum(weights[:, None, None] * features, axis=0)
    cam = np.maximum(cam, 0)
    return cam

cam1 = generate_cam_for_class(image, class1)
cam2 = generate_cam_for_class(image, class2)

# Normalize and compute differential
cam1_norm = (cam1 - cam1.min()) / (cam1.max() - cam1.min() + 1e-8)
cam2_norm = (cam2 - cam2.min()) / (cam2.max() - cam2.min() + 1e-8)
diff_cam = cam1_norm - cam2_norm

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_np)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(create_overlay(img_np, cam1, alpha=0.5))
axes[1].set_title(f'{paths_config["classes"][class1]} CAM')
axes[1].axis('off')

axes[2].imshow(create_overlay(img_np, cam2, alpha=0.5))
axes[2].set_title(f'{paths_config["classes"][class2]} CAM')
axes[2].axis('off')

axes[3].imshow(create_overlay(img_np, np.abs(diff_cam), alpha=0.5, colormap='RdBu_r'))
axes[3].set_title('Differential CAM')
axes[3].axis('off')

plt.tight_layout()
plt.show()